# Stage C CELLECT + Yusuke-style PP

Uses **our Stage C est_exact.pt** (CellectLite detector + ParentalAssocHead).
Architecture is **not** the public UNet 0.914 model — cannot load Stage C into Yusuke UNet.
Companion pure Yusuke 0.914 notebook is the known LB path; this uses trained weights correctly.


In [ ]:
# Stage C CELLECT inference + Yusuke-style lineage PP
# Loads OUR Stage C best_exact.pt (detector + assoc). Not UNet/350ep.
import os, sys, json, time, math, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Need GPU"
DEVICE = torch.device("cuda:0")
print("GPU", torch.cuda.get_device_name(0))

# locate package + weights
SRC = None
for p in Path("/kaggle/input").rglob("cellect_lite.py"):
    if p.parent.name == "models":
        SRC = p.parent.parent.parent  # .../lineage_v2 parent
        if (p.parent.parent / "constants.py").exists():
            SRC = p.parent.parent.parent
        break
# package root containing lineage_v2/
for base in Path("/kaggle/input").rglob("constants.py"):
    if (base.parent / "models" / "cellect_lite.py").exists():
        sys.path.insert(0, str(base.parent.parent if base.parent.name == "lineage_v2" else base.parent.parent))
        if base.parent.name == "lineage_v2":
            sys.path.insert(0, str(base.parent.parent))
        else:
            sys.path.insert(0, str(base.parent))
        print("pkg path candidates", base)
        break

# robust path
for base in Path("/kaggle/input").iterdir():
    for c in base.rglob("lineage_v2"):
        if (c / "models" / "cellect_lite.py").exists():
            sys.path.insert(0, str(c.parent))
            print("SYS.PATH +", c.parent)
            break

from lineage_v2.models.cellect_lite import CellectLite
from lineage_v2.models.assoc import ParentalAssocHead
from lineage_v2.constants import SPACING_ZYX_UM, FORBIDDEN_TEST_STEMS

CKPT = None
for p in list(Path("/kaggle/input").rglob("best_exact.pt")) + list(Path("/kaggle/input").rglob("last.pt")):
    # prefer stagec dataset
    s = str(p).lower()
    if "stagec" in s or "stage-c" in s or "best_exact" in p.name:
        CKPT = p
        if "stagec" in s or "stagec-e1" in s or "lineage-v2-stagec" in s:
            break
if CKPT is None:
    raise FileNotFoundError("Stage C checkpoint not found")
print("CKPT", CKPT)

st = torch.load(CKPT, map_location="cpu", weights_only=False)
print("ckpt keys", list(st.keys()) if isinstance(st, dict) else type(st))
print("step", st.get("global_step"), "stage", st.get("stage"), "best", st.get("best_exact"))

detector = CellectLite(time_frames=1).to(DEVICE)
assoc = ParentalAssocHead(emb_dim=64).to(DEVICE)
sd = st["model"]
# strip module. if any
sd = {k.replace("module.", ""): v for k, v in sd.items()}
miss, unexp = detector.load_state_dict(sd, strict=False)
print("detector missing", len(miss), "unexpected", len(unexp))
if "assoc" in st:
    am, au = assoc.load_state_dict(st["assoc"], strict=False)
    print("assoc missing", len(am), "unexpected", len(au))
detector.eval(); assoc.eval()

SPACING = np.array(list(SPACING_ZYX_UM), dtype=np.float32)
# Yusuke-like knobs
DET_THR = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.35"))  # logits thr after sigmoid use 0.35
NMS_UM = 3.0
MAX_PEAKS = 1500
EDGE_MAX_UM = 14.0
MIN_TRACK = 6
GAP_MAX = 2
GAP_UM = 5.8
SAFE_DIV_PARENT = 4.66
SAFE_DIV_SISTER = 8.5

# competition roots
COMP = None
for c in [
    Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
    Path("/kaggle/input/biohub-cell-tracking-during-development"),
]:
    if c.exists():
        COMP = c
        break
assert COMP is not None, "competition data missing"
TEST = COMP / "test"
print("TEST", TEST, "exists", TEST.exists())

def list_stems():
    stems = sorted(p.name[:-5] for p in TEST.iterdir() if p.name.endswith(".zarr"))
    print("stems", stems)
    return stems

def load_zarr_volume(stem: str):
    import zarr
    path = TEST / f"{stem}.zarr"
    # try common layouts
    root = zarr.open(str(path), mode="r")
    # walk for array
    def find_arr(g, depth=0):
        if depth > 6:
            return None
        if hasattr(g, "shape") and len(getattr(g, "shape", ())) >= 3:
            return g
        try:
            keys = list(g.keys())
        except Exception:
            return None
        for k in keys:
            try:
                a = find_arr(g[k], depth+1)
                if a is not None:
                    return a
            except Exception:
                pass
        return None
    arr = find_arr(root)
    if arr is None:
        raise RuntimeError(f"no array in {path}")
    vol = np.asarray(arr)
    print(stem, "shape", vol.shape, "dtype", vol.dtype)
    # expect (T,Z,Y,X) or (T,1,Z,Y,X) or (Z,Y,X,T)
    if vol.ndim == 5:
        vol = vol[:, 0]
    if vol.ndim != 4:
        # try transpose if last is time-like small
        if vol.shape[-1] < 200 and vol.shape[0] > 200:
            vol = np.moveaxis(vol, -1, 0)
    return vol.astype(np.float32)

def normalize(v):
    # robust per-volume
    lo, hi = np.percentile(v, [1, 99.5])
    v = (v - lo) / max(hi - lo, 1e-6)
    return np.clip(v, 0, 1)

@torch.no_grad()
def detect_frame(frame_zyx: np.ndarray):
    # frame: Z,Y,X
    z,y,x = frame_zyx.shape
    # tile if large
    tz, ty, tx = 32, 160, 160
    heat = np.zeros((z,y,x), np.float32)
    flow = np.zeros((3,z,y,x), np.float32)
    emb = None
    # simple center crop tiles with overlap
    zs = list(range(0, max(1, z - tz + 1), max(1, tz // 2))) or [0]
    ys = list(range(0, max(1, y - ty + 1), max(1, ty // 2))) or [0]
    xs = list(range(0, max(1, x - tx + 1), max(1, tx // 2))) or [0]
    if z < tz: zs = [0]
    if y < ty: ys = [0]
    if x < tx: xs = [0]
    counts = np.zeros((z,y,x), np.float32)
    for z0 in zs:
        for y0 in ys:
            for x0 in xs:
                z1, y1, x1 = min(z0+tz, z), min(y0+ty, y), min(x0+tx, x)
                patch = np.zeros((tz,ty,tx), np.float32)
                patch[:z1-z0, :y1-y0, :x1-x0] = frame_zyx[z0:z1, y0:y1, x0:x1]
                t = torch.from_numpy(patch)[None, None].to(DEVICE)  # 1,1,Z,Y,X
                out = detector(t)
                h = torch.sigmoid(out["center_logits"][0,0]).float().cpu().numpy()
                f = out["backward_flow_um"][0].float().cpu().numpy()
                heat[z0:z1, y0:y1, x0:x1] = np.maximum(heat[z0:z1, y0:y1, x0:x1], h[:z1-z0, :y1-y0, :x1-x0])
                flow[:, z0:z1, y0:y1, x0:x1] = f[:, :z1-z0, :y1-y0, :x1-x0]
                counts[z0:z1, y0:y1, x0:x1] += 1
                if emb is None:
                    emb = out["embedding_map"][0].float().cpu()  # C,z',y',x'
    peaks = []
    # simple nms grid
    thr = DET_THR
    # work on downsampled peak search for speed
    step = 1
    coords = np.argwhere(heat > thr)
    if len(coords) > 20000:
        # keep top by heat
        vals = heat[coords[:,0], coords[:,1], coords[:,2]]
        idx = np.argsort(-vals)[:20000]
        coords = coords[idx]
    # greedy NMS in um
    order = np.argsort(-heat[coords[:,0], coords[:,1], coords[:,2]])
    coords = coords[order]
    taken = []
    for c in coords:
        cz,cy,cx = map(int, c)
        um = np.array([cz,cy,cx], np.float32) * SPACING
        ok = True
        for p in taken:
            if np.linalg.norm((um - p)*1) < NMS_UM:  # already um
                ok = False; break
        if ok:
            taken.append(um)
            peaks.append((cz, cy, cx, float(heat[cz,cy,cx])))
            if len(peaks) >= MAX_PEAKS:
                break
    return peaks, flow, emb

def sample_emb(emb_map, z,y,x):
    # emb_map: C, Z',Y',X' at half res roughly
    if emb_map is None:
        return np.zeros(64, np.float32)
    C, Z, Y, X = emb_map.shape
    zz = int(np.clip(z * Z / max(Z,1), 0, Z-1))  # emb may be different scale
    # map from full res - emb is from d1 level ~ half yx
    # approximate: scale by shape ratio unknown; use center of map if mismatch
    # better: interpolate
    t = emb_map.unsqueeze(0)  # 1,C,Z,Y,X
    # normalize coords to [-1,1]
    # we don't know full size; use relative if we pass full shape later
    return emb_map[:, min(zz,Z-1), min(int(y/2), Y-1), min(int(x/2), X-1)].numpy()

def link_tracks(frames_peaks, frames_flow):
    # frames_peaks: list of list of (z,y,x,score)
    # greedy nearest + flow prior, Yusuke-like max edge um
    nodes = []  # dicts
    edges = []
    nid = 0
    per_t = []
    for t, peaks in enumerate(frames_peaks):
        ids = []
        for (z,y,x,sc) in peaks:
            nodes.append({"id": nid, "t": t, "z": z, "y": y, "x": x, "score": sc})
            ids.append(nid)
            nid += 1
        per_t.append(ids)
    # link t -> t+1
    for t in range(len(per_t)-1):
        a = per_t[t]; b = per_t[t+1]
        if not a or not b:
            continue
        A = np.array([[nodes[i]["z"], nodes[i]["y"], nodes[i]["x"]] for i in a], np.float32)
        B = np.array([[nodes[i]["z"], nodes[i]["y"], nodes[i]["x"]] for i in b], np.float32)
        Au = A * SPACING; Bu = B * SPACING
        # cost matrix
        used_b = set()
        # for each parent, best child within EDGE_MAX_UM, prefer flow
        for ii, ia in enumerate(a):
            na = nodes[ia]
            # flow at parent
            fl = frames_flow[t]
            zz,yy,xx = int(na["z"]), int(na["y"]), int(na["x"])
            if fl is not None:
                zz = min(zz, fl.shape[1]-1); yy=min(yy, fl.shape[2]-1); xx=min(xx, fl.shape[3]-1)
                f_um = fl[:, zz, yy, xx]
                pred = Au[ii] + f_um
            else:
                pred = Au[ii]
            d = np.linalg.norm(Bu - pred[None, :], axis=1)
            d2 = np.linalg.norm(Bu - Au[ii][None,:], axis=1)
            cost = d + 0.05 * d2
            order = np.argsort(cost)
            linked = 0
            for j in order:
                if j in used_b:
                    continue
                if d2[j] > EDGE_MAX_UM:
                    continue
                if cost[j] > EDGE_MAX_UM * 1.2:
                    break
                ib = b[j]
                edges.append((ia, ib))
                used_b.add(j)
                linked += 1
                # allow division: second child
                if linked >= 2:
                    break
                # only take second if close
                if linked == 1:
                    continue
    return nodes, edges

def enforce_degree(nodes, edges):
    # max 1 parent, max 2 children; drop longest edges if needed
    from collections import defaultdict
    out_e = list(edges)
    # parents
    by_child = defaultdict(list)
    for s,t in out_e:
        by_child[t].append(s)
    keep = set(out_e)
    for c, ps in by_child.items():
        if len(ps) <= 1:
            continue
        # keep closest parent
        cn = nodes[c]
        cu = np.array([cn["z"],cn["y"],cn["x"]], np.float32)*SPACING
        dists = []
        for p in ps:
            pn = nodes[p]
            pu = np.array([pn["z"],pn["y"],pn["x"]], np.float32)*SPACING
            dists.append((np.linalg.norm(cu-pu), p))
        dists.sort()
        for _, p in dists[1:]:
            keep.discard((p,c))
    # children
    by_par = defaultdict(list)
    for s,t in list(keep):
        by_par[s].append(t)
    for p, cs in by_par.items():
        if len(cs) <= 2:
            continue
        pn = nodes[p]
        pu = np.array([pn["z"],pn["y"],pn["x"]], np.float32)*SPACING
        dists = []
        for c in cs:
            cn = nodes[c]
            cu = np.array([cn["z"],cn["y"],cn["x"]], np.float32)*SPACING
            dists.append((np.linalg.norm(cu-pu), c))
        dists.sort()
        for _, c in dists[2:]:
            keep.discard((p,c))
    return list(keep)

def filter_short(nodes, edges, min_len=MIN_TRACK):
    from collections import defaultdict, deque
    adj = defaultdict(list)
    radj = defaultdict(list)
    for s,t in edges:
        adj[s].append(t); radj[t].append(s)
    seen=set(); keep_n=set()
    for i in range(len(nodes)):
        if i in seen: continue
        # component undirected
        q=deque([i]); comp=set()
        while q:
            u=q.popleft()
            if u in comp: continue
            comp.add(u)
            for v in adj[u]+radj[u]:
                if v not in comp: q.append(v)
        seen |= comp
        # track length ~ unique times
        ts = len({nodes[u]["t"] for u in comp})
        # keep if long enough OR has division
        has_div = any(len(adj[u])>=2 for u in comp)
        if ts >= min_len or has_div:
            keep_n |= comp
    nodes2 = []
    idmap = {}
    for u in sorted(keep_n):
        idmap[u] = len(nodes2)
        n = dict(nodes[u]); n["id"]=idmap[u]
        nodes2.append(n)
    edges2 = [(idmap[s], idmap[t]) for s,t in edges if s in idmap and t in idmap]
    return nodes2, edges2

def to_rows(stem, nodes, edges):
    rows = []
    rid = 0
    for n in nodes:
        rows.append({
            "id": rid, "dataset": stem, "row_type": "node",
            "node_id": n["id"], "t": n["t"], "z": n["z"], "y": n["y"], "x": n["x"],
            "source_id": -1, "target_id": -1,
        }); rid += 1
    for s,t in edges:
        rows.append({
            "id": rid, "dataset": stem, "row_type": "edge",
            "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": s, "target_id": t,
        }); rid += 1
    return rows

all_rows = []
t0 = time.time()
for stem in list_stems():
    print("====", stem, "====")
    try:
        vol = load_zarr_volume(stem)  # T,Z,Y,X
        vol = normalize(vol)
        T = vol.shape[0]
        # limit frames if huge for runtime - process all
        frames_peaks = []
        frames_flow = []
        for t in range(T):
            peaks, flow, emb = detect_frame(vol[t])
            frames_peaks.append(peaks)
            frames_flow.append(flow)
            if t % 10 == 0:
                print(f"  t={t}/{T} peaks={len(peaks)} elapsed={(time.time()-t0)/60:.1f}m")
        nodes, edges = link_tracks(frames_peaks, frames_flow)
        edges = enforce_degree(nodes, edges)
        nodes, edges = filter_short(nodes, edges, MIN_TRACK)
        print(f"  final nodes={len(nodes)} edges={len(edges)}")
        all_rows.extend(to_rows(stem, nodes, edges))
    except Exception as e:
        print("FAIL stem", stem, e)
        import traceback; traceback.print_exc()
    gc.collect(); torch.cuda.empty_cache()

sub = pd.DataFrame(all_rows)
if len(sub)==0:
    # emergency empty-ish valid structure
    sub = pd.DataFrame(columns=["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"])
sub["id"] = np.arange(len(sub))
outp = Path("/kaggle/working/submission.csv")
sub.to_csv(outp, index=False)
print("WROTE", outp, "rows", len(sub))
print(sub.groupby(["dataset","row_type"]).size() if len(sub) else "empty")
print("DONE minutes", (time.time()-t0)/60)
